# Retards et annulations de vols aux États-Unis en 2015

Ce notebook vérifie la qualité des données, puis cherche ce qui explique les retards et les annulations.

In [5]:
import pandas as pd

flights = pd.read_csv("../data/raw/flights.csv", low_memory=False)
len(flights)

5819079

## 1. Nettoyage

### Pourquoi certains vols n'ont-ils pas de retard au départ ?

Hypothèse : ces vols ont été annulés, donc ils ne sont jamais partis. Si c'est vrai, ces cases vides ne sont pas des erreurs, et il ne faut pas les remplacer par 0.

In [6]:
pd.crosstab(flights["DEPARTURE_DELAY"].isna(), flights["CANCELLED"])

CANCELLED,0,1
DEPARTURE_DELAY,,
False,5729195,3731
True,0,86153


**Constat :** tous les retards vides correspondent à des vols annulés. On ne les remplace donc pas par 0.

### Faut-il garder les vols annulés qui ont un retard au départ ?

**Choix : on les garde.** Un long retard au départ pourrait précéder une annulation : les retirer ferait disparaître ce signal. On compare donc leur retard à celui des autres vols.

In [7]:
flights.groupby("CANCELLED")["DEPARTURE_DELAY"].describe().round(1)

,count,mean,std,min,25%,50%,75%,max
CANCELLED,,,,,,,,
0,5729195.0,9.3,37.0,-82.0,-5.0,-2.0,7.0,1988.0
1,3731.0,57.5,95.5,-25.0,-2.0,21.0,83.0,1274.0


In [8]:
avec_annules = flights["DEPARTURE_DELAY"].mean()
sans_annules = flights.loc[flights["CANCELLED"] == 0, "DEPARTURE_DELAY"].mean()
round(avec_annules, 2), round(sans_annules, 2)

(np.float64(9.37), np.float64(9.34))

### Quand les causes de retard sont-elles renseignées ?

Hypothèse : les 5 colonnes de causes ne sont remplies que pour les vols arrivés avec au moins 15 minutes de retard.

In [ ]:
causes = ["AIR_SYSTEM_DELAY", "SECURITY_DELAY", "AIRLINE_DELAY", "LATE_AIRCRAFT_DELAY", "WEATHER_DELAY"]

cause_renseignee = flights[causes].notna().all(axis=1)
retard_15_min = flights["ARRIVAL_DELAY"] >= 15

pd.crosstab(retard_15_min, cause_renseignee)

### La somme des causes correspond-elle au retard à l'arrivée ?

In [ ]:
vols_avec_cause = flights[cause_renseignee]
somme_causes = vols_avec_cause[causes].sum(axis=1)

(somme_causes == vols_avec_cause["ARRIVAL_DELAY"]).mean()

**Constat :** les causes sont renseignées pour tous les vols arrivés avec au moins 15 minutes de retard, et seulement pour eux. Leur somme est toujours égale au retard à l'arrivée.

**Conséquence :** on peut calculer la part de chaque cause, mais uniquement pour ces vols. Une case vide veut dire « cause non renseignée », pas « 0 minute ».

### Les codes d'aéroport des vols existent-ils dans la liste des aéroports ?

Hypothèse : les codes d'octobre sont écrits autrement, donc ce mois-là ne se relie pas à `airports.csv`.

In [ ]:
airports = pd.read_csv("../data/raw/airports.csv")

origine_connue = flights["ORIGIN_AIRPORT"].isin(airports["IATA_CODE"])
destination_connue = flights["DESTINATION_AIRPORT"].isin(airports["IATA_CODE"])

pd.crosstab(flights["MONTH"], [origine_connue, destination_connue])

In [ ]:
flights.loc[~origine_connue, "ORIGIN_AIRPORT"].value_counts().head()

In [ ]:
airports[airports[["LATITUDE", "LONGITUDE"]].isna().any(axis=1)]

**Constat :** tous les vols d'octobre ont des codes à 5 chiffres, alors que les 11 autres mois utilisent des codes à 3 lettres. Sans correction, octobre disparaît des analyses par aéroport.

### Traduire les codes d'octobre

**Choix :** on récupère ces vols plutôt que de les exclure, car ils représentent 8,4 % des données. Le bureau américain des statistiques de transport (BTS) publie deux tables qui donnent le nom de chaque aéroport : l'une par numéro, l'autre par code à lettres. Le nom sert de lien entre les deux.

In [9]:
# Ces fichiers ne sont pas en UTF-8. keep_default_na=False évite que le code "NA" soit lu comme une case vide.
table_numeros = pd.read_csv("../data/raw/L_AIRPORT_ID.csv", dtype=str, encoding="latin-1", keep_default_na=False)
table_lettres = pd.read_csv("../data/raw/L_AIRPORT.csv", dtype=str, encoding="latin-1", keep_default_na=False)

table_numeros[table_numeros["Code"] == "10397"]

,Code,Description
373,10397,"Atlanta, GA: Hartsfield-Jackson Atlanta Intern..."


In [10]:
est_octobre = flights["MONTH"] == 10
colonnes_aeroport = ["ORIGIN_AIRPORT", "DESTINATION_AIRPORT"]

codes_octobre = pd.unique(flights.loc[est_octobre, colonnes_aeroport].values.ravel())
codes_autres_mois = pd.unique(flights.loc[~est_octobre, colonnes_aeroport].values.ravel())

correspondance = table_numeros[table_numeros["Code"].isin(codes_octobre)].merge(
    table_lettres[table_lettres["Code"].isin(codes_autres_mois)],
    on="Description",
    suffixes=("_numero", "_lettres"),
)

numeros_sans_traduction = set(codes_octobre) - set(correspondance["Code_numero"])
len(codes_octobre), len(correspondance), numeros_sans_traduction

(307, 306, {'14027'})

### Le code 14027 n'a pas de traduction : lequel est-ce ?

Une route garde la même distance toute l'année. On compare donc les routes de 14027 à celles de PBI (Palm Beach).

In [ ]:
traduction = correspondance.set_index("Code_numero")["Code_lettres"]

routes_14027 = flights[flights["ORIGIN_AIRPORT"] == "14027"]
distances_14027 = routes_14027.groupby(routes_14027["DESTINATION_AIRPORT"].map(traduction))["DISTANCE"].median()

routes_pbi = flights[flights["ORIGIN_AIRPORT"] == "PBI"]
distances_pbi = routes_pbi.groupby("DESTINATION_AIRPORT")["DISTANCE"].median()

pd.DataFrame({"14027": distances_14027, "PBI": distances_pbi.reindex(distances_14027.index)})

**Constat :** les distances sont identiques sur toutes les routes. On ajoute donc 14027 = PBI à la main, puis on traduit les codes.

In [ ]:
traduction["14027"] = "PBI"

for colonne in colonnes_aeroport:
    flights[colonne] = flights[colonne].map(traduction).fillna(flights[colonne])

flights["ORIGIN_AIRPORT"].isin(airports["IATA_CODE"]).mean(), flights["DESTINATION_AIRPORT"].isin(airports["IATA_CODE"]).mean()

### La traduction est-elle juste ?

On compare la distance de chaque vol d'octobre à la distance de la même route les autres mois. Une mauvaise traduction créerait de grands écarts.

In [ ]:
distance_habituelle = (
    flights[~est_octobre]
    .groupby(colonnes_aeroport)["DISTANCE"]
    .median()
    .rename("DISTANCE_HABITUELLE")
)

vols_octobre = flights[est_octobre].join(distance_habituelle, on=colonnes_aeroport)
ecart = (vols_octobre["DISTANCE"] - vols_octobre["DISTANCE_HABITUELLE"]).abs()

ecart.value_counts(dropna=False).sort_index()

### Combien de vols sont touchés par les aéroports sans coordonnées ?

In [ ]:
sans_coordonnees = airports.loc[airports["LATITUDE"].isna(), "IATA_CODE"]

(flights["ORIGIN_AIRPORT"].isin(sans_coordonnees) | flights["DESTINATION_AIRPORT"].isin(sans_coordonnees)).sum()

**Constat :** 3 aéroports (ECP, PBG, UST) n'ont pas de coordonnées, ce qui touche 10 020 vols, soit 0,17 %.

**Choix :** on garde ces vols dans les chiffres. Ces aéroports n'apparaîtront pas sur la carte, et on le précisera sous celle-ci.

### Les horaires sont-ils utilisables ?

Les horaires sont écrits HHMM : 1430 veut dire 14 h 30, et 5 veut dire 00 h 05. On vérifie d'abord que tous les horaires sont valides.

In [ ]:
colonnes_horaires = ["SCHEDULED_DEPARTURE", "DEPARTURE_TIME", "SCHEDULED_ARRIVAL", "ARRIVAL_TIME"]

pd.DataFrame({
    "minimum": flights[colonnes_horaires].min(),
    "maximum": flights[colonnes_horaires].max(),
    "minutes_invalides": (flights[colonnes_horaires] % 100 >= 60).sum(),
    "egal_a_2400": (flights[colonnes_horaires] == 2400).sum(),
})

**Constat :** aucun horaire n'a de minutes invalides. La valeur 2400 existe pour minuit dans les heures réelles et d'arrivée, mais jamais dans le départ prévu : l'heure extraite de cette colonne reste donc entre 0 et 23.

### Les horaires sont-ils cohérents avec le retard ?

Heure prévue + retard doit donner l'heure réelle de départ. On compte en minutes depuis minuit, et le modulo 1440 gère le passage de minuit.

In [ ]:
def en_minutes(horaire):
    return horaire // 100 * 60 + horaire % 100


depart_calcule = en_minutes(flights["SCHEDULED_DEPARTURE"]) + flights["DEPARTURE_DELAY"]
depart_reel = en_minutes(flights["DEPARTURE_TIME"])
a_un_depart = flights["DEPARTURE_TIME"].notna()

(((depart_calcule - depart_reel) % 1440 == 0)[a_un_depart]).mean()

**Constat :** un seul vol sur 5 732 926 ne respecte pas le calcul. C'est un vol annulé du 24 décembre, prévu à 8 h 04 et parti à 11 h 03, avec un retard indiqué de 1 minute. **Choix :** on le garde, car il ne change aucun résultat.

### Créer l'heure de départ prévue

On garde uniquement l'heure, de 0 à 23, pour comparer les moments de la journée. On utilise l'heure prévue, car elle existe pour tous les vols, même annulés.

In [ ]:
flights["HEURE_DEPART_PREVUE"] = flights["SCHEDULED_DEPARTURE"] // 100

flights.groupby("HEURE_DEPART_PREVUE")["DEPARTURE_DELAY"].agg(["count", "mean", "median"]).round(1)

**Constat :** le retard moyen passe d'environ 2 minutes à 5-6 h à 15 minutes vers 18-20 h. Entre 0 h et 4 h, il y a trop peu de vols pour conclure avec la même confiance.

**Hypothèse à vérifier :** les retards s'accumulent au fil de la journée, car un avion en retard repart en retard.

### Les valeurs extrêmes sont-elles possibles ?

On regarde d'abord le plus petit et le plus grand chiffre de chaque colonne. Une valeur impossible serait par exemple une distance nulle ou un temps de vol négatif.

In [ ]:
colonnes_durees = ["DEPARTURE_DELAY", "ARRIVAL_DELAY", "AIR_TIME", "ELAPSED_TIME", "DISTANCE"]

flights[colonnes_durees].describe().round(1)

**Constat :** aucune valeur impossible dans ces colonnes : la distance et le temps de vol sont toujours positifs. Un retard négatif veut dire que le vol est en avance.

### Les plus gros retards sont-ils de vraies données ?

Si le plus gros retard était une erreur de saisie, l'avion ne serait pas arrivé en retard d'autant. On compare donc le retard au départ et à l'arrivée.

In [ ]:
colonnes_vol = ["MONTH", "DAY", "AIRLINE", "ORIGIN_AIRPORT", "DESTINATION_AIRPORT", "SCHEDULED_DEPARTURE", "DEPARTURE_TIME", "DEPARTURE_DELAY", "ARRIVAL_DELAY"]

flights.nlargest(5, "DEPARTURE_DELAY")[colonnes_vol]

**Constat :** le plus gros retard est de 1 988 minutes, soit plus de 33 heures. Il est crédible : le vol prévu à 7 h 00 est parti à 16 h 08 le lendemain et il est arrivé avec 1 971 minutes de retard.

### Combien de vols ont un très gros retard, et pèsent-ils sur la moyenne ?

On compare la moyenne avec et sans les vols partis avec plus de 3 heures de retard.

In [ ]:
retard = flights["DEPARTURE_DELAY"]
plus_de_3h = retard > 180

plus_de_3h.sum(), plus_de_3h.mean(), retard.mean(), retard[~plus_de_3h].mean(), retard.quantile([0.5, 0.9, 0.99])

**Constat :** 48 107 vols, soit 0,8 %, partent avec plus de 3 heures de retard. Sans eux, la moyenne passe de 9,4 à 7,1 minutes.

**Choix :** on garde ces vols, car ce sont de vrais retards. Comme ils tirent la moyenne vers le haut, on ne s'appuiera pas sur elle seule : on regardera aussi la médiane et la part de vols en retard.

### Y a-t-il des vols en double ?

Un doublon compterait un même vol deux fois. On cherche d'abord les lignes entièrement identiques, puis les lignes qui décrivent le même vol avec des chiffres différents : même jour, même compagnie, même numéro de vol et même heure prévue au même aéroport.

In [ ]:
colonnes_identite = ["MONTH", "DAY", "AIRLINE", "FLIGHT_NUMBER", "ORIGIN_AIRPORT", "SCHEDULED_DEPARTURE"]

flights.duplicated().sum(), flights.duplicated(subset=colonnes_identite).sum()

In [ ]:
flights[flights.duplicated(subset=colonnes_identite, keep=False)].sort_values(colonnes_identite)[colonnes_identite + ["DESTINATION_AIRPORT", "TAIL_NUMBER", "DEPARTURE_DELAY", "CANCELLED"]].head(10)

**Constat :** aucune ligne n'est entièrement identique. Une seule paire partage la même identité : le vol AA 803 du 29 août, au départ de STT à 14 h 35. Ses deux lignes ont des destinations différentes (CLT et SJU) et sont toutes les deux annulées.

**Choix :** on ne supprime aucune ligne, car rien ne prouve que ce soit un doublon.